#  TP Synthèse — Prédire la progression d'une maladie

> **Modalité :** Individuel  
> **Rendu :** Ce notebook complété, avec toutes les cellules exécutées et les questions répondues

---

###  Contexte métier

Vous travaillez pour un laboratoire pharmaceutique qui cherche à **prédire la progression d'une maladie** chez des patients diabétiques un an après le diagnostic.

Le dataset contient **442 patients** décrits par **10 mesures cliniques** (âge, IMC, tension artérielle, marqueurs sanguins...). La cible `y` est un indice de progression de la maladie mesuré un an plus tard.

> L'objectif n'est pas simplement d'obtenir un bon score — c'est de **comprendre votre modèle**, de le **valider rigoureusement** et d'être capable de **justifier vos choix** à un médecin qui n'a aucune connaissance en ML.

---

###  Compétences mobilisées

| Partie | Contenu | Cours |
|--------|---------|-------|
| 1 | Exploration et régression simple | Cours 1 |
| 2 | Régression multiple, Pipeline, VIF | Cours 1 |
| 3 | Métriques avancées et analyse des résidus | Cours 2 |
| 4 | Cross-validation et robustesse | Cours 2 |
| 5 | Régularisation Ridge et Lasso | Cours 2 |
| Synthèse | Tableau comparatif final + recommandation | Cours 1 + 2 |

---

###  Livrables attendus

- [ ] Toutes les cellules `✏️ VOTRE CODE ICI` complétées et exécutées
- [ ] Les **9 graphiques** demandés, affichés avec titres et axes labellisés
- [ ] Les **questions ouvertes** (marquées `❓`) répondues en phrases complètes
- [ ] Le **tableau comparatif final** renseigné avec vos chiffres réels
- [ ] La **recommandation finale** rédigée (cellule Synthèse)

---

>  **Conseil de méthode :** Lisez la consigne de chaque partie en entier avant de commencer à coder.  
> Certaines parties utilisent des variables définies dans les parties précédentes.


---
##  Setup — À exécuter en premier, sans modification

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_diabetes
from sklearn.linear_model import (LinearRegression, Ridge, Lasso,
                                   RidgeCV, LassoCV)
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      cross_validate, KFold, learning_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.rcParams.update({
    'figure.figsize': (10, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
np.random.seed(42)

# ── Chargement du dataset ────────────────────────────────────────────────────
raw = load_diabetes(as_frame=True)
X   = raw.data.copy()
y   = raw.target.copy()
y.name = 'progression'

# Renommer les colonnes pour plus de lisibilité
X.columns = ['age', 'sexe', 'imc', 'tension', 'cholesterol_total',
             'ldl', 'hdl', 'cholesterol_ratio', 'glycemie', 'insuline']

# Enrichir le dataset avec 15 features bruit (pour la partie régularisation)
np.random.seed(99)
X_bruit = pd.DataFrame(
    np.random.randn(len(X), 15),
    columns=[f'bruit_{i+1:02d}' for i in range(15)],
    index=X.index
)
X_enrichi = pd.concat([X, X_bruit], axis=1)

print("=" * 55)
print(f"  Dataset : Diabetes Progression (sklearn)")
print(f"  Patients : {X.shape[0]}   |   Features : {X.shape[1]}")
print(f"  Cible   : progression de la maladie (1 an après)")
print("=" * 55)
print()
print("Aperçu des 3 premiers patients :")
print(pd.concat([X.head(3), y.head(3)], axis=1).to_string())
print()
print("Note : les features sont déjà standardisées dans le dataset original.")
print("Pour les exercices de standardisation, nous travaillerons avec des copies non-standardisées.")
print()

# Dé-standardiser pour les exercices (recréer des valeurs "réalistes")
X_raw = X.copy()
scales = {'age': 15, 'sexe': 0.5, 'imc': 5, 'tension': 15,
          'cholesterol_total': 40, 'ldl': 40, 'hdl': 15,
          'cholesterol_ratio': 20, 'glycemie': 30, 'insuline': 80}
means  = {'age': 48, 'sexe': 0, 'imc': 26, 'tension': 94,
          'cholesterol_total': 189, 'ldl': 115, 'hdl': 49,
          'cholesterol_ratio': 4.1, 'glycemie': 91, 'insuline': 80}
for col in X_raw.columns:
    X_raw[col] = X_raw[col] * scales[col] + means[col]

print("\nFeatures et descriptions :")
descriptions = {
    'age':               'Âge du patient (années)',
    'sexe':              'Sexe (encodé numériquement)',
    'imc':               'Indice de masse corporelle (kg/m²)',
    'tension':           'Pression artérielle moyenne (mmHg)',
    'cholesterol_total': 'Taux de cholestérol total (mg/dL)',
    'ldl':               'LDL cholestérol — « mauvais » (mg/dL)',
    'hdl':               'HDL cholestérol — « bon » (mg/dL)',
    'cholesterol_ratio': 'Ratio LDL/HDL',
    'glycemie':          'Glycémie à jeun (mg/dL)',
    'insuline':          'Taux d\'insuline sérique (μIU/mL)',
}
for col in X_raw.columns:
    print(f"  {col:<20} → {descriptions[col]}")

print(f"\nVariable cible (progression) :")
print(f"  Moyenne : {y.mean():.1f}   Écart-type : {y.std():.1f}   Min : {y.min():.0f}   Max : {y.max():.0f}")
print(" Setup OK — Vous pouvez commencer !")


  Dataset : Diabetes Progression (sklearn)
  Patients : 442   |   Features : 10
  Cible   : progression de la maladie (1 an après)

Aperçu des 3 premiers patients :
        age      sexe       imc   tension  cholesterol_total       ldl       hdl  cholesterol_ratio  glycemie  insuline  progression
0  0.038076  0.050680  0.061696  0.021872          -0.044223 -0.034821 -0.043401          -0.002592  0.019907 -0.017646        151.0
1 -0.001882 -0.044642 -0.051474 -0.026328          -0.008449 -0.019163  0.074412          -0.039493 -0.068332 -0.092204         75.0
2  0.085299  0.050680  0.044451 -0.005670          -0.045599 -0.034194 -0.032356          -0.002592  0.002861 -0.025930        141.0

Note : les features sont déjà standardisées dans le dataset original.
Pour les exercices de standardisation, nous travaillerons avec des copies non-standardisées.


Features et descriptions :
  age                  → Âge du patient (années)
  sexe                 → Sexe (encodé numériquement)
  imc   

---
## Partie 1 — Exploration et régression linéaire simple

---

### 1.1 — Explorer la variable cible

Avant tout modèle, il faut comprendre ce qu'on cherche à prédire.

**Consigne :** Produisez une figure avec **deux sous-graphiques** :
- **Gauche** : histogramme de la variable `y` (progression) avec la courbe normale théorique superposée et une ligne verticale pour la moyenne
- **Droite** : boxplot de `y` annoté avec Q1, médiane, Q3 et les outliers identifiés

Ajoutez dans le titre global : le nombre de patients, la moyenne et l'écart-type de `y`.

> **Rappel :** Pour superposer une courbe normale : `stats.norm.pdf(x_range, mu, sigma)`




In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 1.1

La distribution de `y` est-elle normale ? Symétrique ? Y a-t-il des valeurs extrêmes visibles ?  
Quelle conséquence cela peut-il avoir sur les métriques MSE et MAE ?

> ✏️ *Répondez ici.*

---

### 1.2 — Identifier la meilleure feature pour une régression simple

**Consigne :** Calculez la corrélation de Pearson entre chaque feature de `X_raw` et `y`.  
Affichez un **barplot horizontal** des corrélations triées par valeur absolue décroissante.  
Colorez en bleu les corrélations positives, en orange les négatives.

Identifiez la feature la plus corrélée avec `y`. C'est elle que vous utiliserez pour la régression simple.



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 1.2

Quelle est la feature la plus corrélée avec `y` ? Quelle est la valeur de r ?  
Est-ce que cette corrélation vous surprend d'un point de vue médical ?  
Peut-on conclure que cette feature *cause* la progression de la maladie ? Justifiez.

> ✏️ *Répondez ici.*

---

### 1.3 — Régression linéaire simple : descente de gradient vs sklearn

Vous allez implémenter la régression simple de **deux façons** pour comprendre ce qui se passe "sous le capot".

**Consigne :**

**Étape A — sklearn :**
1. Splittez `X_raw` et `y` en train (80%) et test (20%), `random_state=42`
2. Entraînez une `LinearRegression` sur la seule feature la plus corrélée (trouvée en 1.2), **uniquement sur X_train**
3. Calculez RMSE_train, RMSE_test, R²_train, R²_test
4. Affichez les coefficients β₀ et β₁ et leur interprétation métier (ex : « +1 unité de IMC → +X unités de progression »)

**Étape B — Vérification manuelle :**
Calculez les coefficients OLS à la main avec les formules :
- `β₁ = Cov(x, y) / Var(x)`
- `β₀ = mean(y) - β₁ * mean(x)`

Comparez avec les valeurs sklearn. Elles doivent être identiques (ou très proches).


In [ ]:
# ✏️ VOTRE CODE ICI


**Graphique 1 — Droite de régression + résidus**

Produisez un graphique en 3 sous-plots côte à côte :
- **Gauche** : scatter des données de train + droite de régression (équation dans le titre)
- **Centre** : scatter y_réel vs ŷ sur le test, avec la droite parfaite (y=x) — même échelle sur les deux axes
- **Droite** : histogramme des résidus sur le test + courbe normale théorique



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 1.3

Comparez RMSE_train et RMSE_test. Sont-ils proches ?  
Le R² obtenu vous semble-t-il satisfaisant pour un contexte médical ? Justifiez en pensant à l'utilisation concrète.

> ✏️ *Répondez ici.*


---
## Partie 2 — Régression linéaire multiple, multicolinéarité et Pipeline

---

### 2.1 — Matrice de corrélation et multicolinéarité

**Consigne :** Produisez la **heatmap de corrélation** de toutes les features de `X_raw`.  
Identifiez visuellement les paires de features fortement corrélées (|r| > 0.7).

> **Rappel :** `sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)`



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 2.1

Identifiez au moins deux paires de features très corrélées entre elles.  
Pourquoi cela peut-il être problématique en régression multiple ?  
Quel indicateur va-t-on utiliser pour le quantifier ?

> ✏️ *Répondez ici.*

---

### 2.2 — Calcul et interprétation du VIF

**Consigne :** Implémentez une fonction `compute_vif(X_df)` qui calcule le VIF pour chaque feature :

```
VIF(xⱼ) = 1 / (1 − R²ⱼ)
où R²ⱼ = R² de la régression de xⱼ sur toutes les autres features
```

Affichez les VIF triés, avec un code couleur :
- 🟢 VIF < 5 : OK
- 🟠 VIF entre 5 et 10 : à surveiller  
- 🔴 VIF > 10 : multicolinéarité sévère



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 2.2

Quelles features présentent un VIF problématique ?  
Proposez trois stratégies différentes pour gérer la multicolinéarité détectée.  
Laquelle appliquerez-vous dans la suite ? Justifiez.

> ✏️ *Répondez ici.*

---

### 2.3 — Régression multiple avec Pipeline

**Consigne :** Construisez un Pipeline `StandardScaler → LinearRegression` et entraînez-le sur `X_train`.

> ⚠️ **Rappel critique :** Le scaler doit être fitté **uniquement sur X_train**. Un Pipeline garantit cela automatiquement.

Calculez et affichez :
- RMSE_train, RMSE_test, R²_train, R²_test
- Le gain en RMSE par rapport à la régression simple (Partie 1)
- Le nombre de degrés de liberté résiduels : `n_train - p - 1`



In [ ]:
# ✏️ VOTRE CODE ICI


**Graphique 2 — Importance et sens des features**

Produisez un **barplot horizontal** des coefficients standardisés (récupérés depuis `pipeline.named_steps['linearregression'].coef_`).  
- Colorez en bleu les effets positifs sur la progression, en orange les effets négatifs
- Ajoutez des annotations avec les valeurs numériques
- Titre : "Coefficients de la régression multiple — dataset Diabetes"

> 💡 Un coefficient standardisé élevé (en valeur absolue) signifie que cette feature est un meilleur prédicteur que les autres, *toutes choses égales par ailleurs*.



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 2.3

Comparez les résultats : régression simple (1 feature) vs régression multiple (10 features).  
Le gain de R² justifie-t-il l'ajout de 9 features supplémentaires ?  
Quelle feature a l'effet le plus fort sur la progression ? Cela est-il cohérent avec la corrélation calculée en 1.2 ?

> ✏️ *Répondez ici.*


---
## Partie 3 — Métriques avancées et diagnostic des résidus

---

### 3.1 — Comparaison des métriques

**Consigne :** Calculez sur le jeu de test les 4 métriques suivantes pour votre modèle multiple :
- MSE, RMSE, MAE, R², R² ajusté

Formulez l'interprétation **métier** de chaque métrique :  
*Ex : « Notre modèle prédit la progression avec une erreur typique de ±XX points d'indice »*

Calculez aussi le **R² ajusté** à la main :

```
R²_adj = 1 − (1 − R²) × (n − 1) / (n − p − 1)
```

Comparez R² et R² ajusté. Sont-ils proches ?



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 3.1

Quelle métrique utiliseriez-vous en priorité pour présenter la performance à un médecin ? Pourquoi ?  
Si un modèle concurrent proposait RMSE = 53.0 mais commettait parfois des erreurs de +200 sur des cas extrêmes, quelle métrique révèlerait ce problème que RMSE ne montre pas ?

> ✏️ *Répondez ici.*

---

### 3.2 — Diagnostic complet des résidus

Les résidus d'un bon modèle de régression doivent satisfaire 4 conditions.  
Votre mission est de vérifier **chacune des quatre**.

**Consigne :** Produisez une figure avec **4 sous-graphiques** (disposition 2×2) :

**[1] Résidus vs Valeurs prédites**  
- Scatter de `(ŷ, résidu)` avec une ligne horizontale en 0  
- Superposer une courbe de tendance locale (polynôme degré 2 avec `np.polyfit`)  
- Ce graphique teste : **homoscédasticité** et **linéarité**

**[2] Scale-Location (√|résidus| vs ŷ)**  
- Scatter de `(ŷ, √|résidu|)` avec une droite de tendance  
- Ce graphique teste : **homoscédasticité** (variance constante)

**[3] QQ-plot des résidus**  
- Utiliser `stats.probplot(residus, plot=ax)`  
- Ce graphique teste : **normalité** de la distribution des résidus

**[4] Résidus vs ordre d'observation**  
- Scatter de `(index, résidu)` avec ligne en 0  
- Ce graphique teste : **indépendance** des résidus (pas d'autocorrélation)

> 💡 `residus_test = y_test.values - y_pred_test`  
> Dans le titre de chaque sous-graphique, indiquez ce que vous cherchez et ce que vous observez.


In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 3.2 — Tableau de diagnostic

Complétez le tableau suivant avec vos observations :

| Graphique | Condition testée | Ce que vous observez | Diagnostic : OK / Problème |
|-----------|-----------------|----------------------|---------------------------|
| [1] Résidus vs ŷ | Homoscédasticité + linéarité | ??? | ??? |
| [2] Scale-Location | Homoscédasticité | ??? | ??? |
| [3] QQ-plot | Normalité | ??? | ??? |
| [4] Résidus vs index | Indépendance | ??? | ??? |

> ✏️ *Complétez le tableau, puis rédigez une synthèse de 2–3 phrases.*

---

### 3.3 — Résidus en fonction de chaque feature

**Consigne :** Pour chaque feature, tracez les résidus en fonction de la valeur de la feature (scatter + tendance locale).  
Organisez les 10 graphiques en **2 lignes × 5 colonnes**.  
Calculez la corrélation `r(feature, résidu)` et affichez-la dans le titre de chaque sous-graphique.

> ⚠️ Un `r` non nul entre une feature et les résidus indique que la relation est **non-linéaire** — le modèle laisse de l'information dans les résidus.



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 3.3

Quelle feature montre la corrélation résidu la plus forte ?  
Que suggère-t-elle sur la relation entre cette feature et la progression de la maladie ?  
Quelle transformation ou enrichissement du modèle pourrait corriger ce problème ?

> ✏️ *Répondez ici.*


---
## Partie 4 — Cross-validation et robustesse de l'évaluation

---

### 4.1 — Comparer un split unique vs K-Fold

**Consigne :** Démontrez la variance d'un seul split en comparant :

**Étape A :** Exécutez 30 évaluations avec 30 `random_state` différents (0 à 29) sur un split 80/20.  
Pour chaque split, calculez le RMSE_test du modèle multiple (Pipeline `StandardScaler + LinearRegression`).  
Tracez un graphique montrant l'évolution du RMSE selon le `random_state`, avec la moyenne et les bandes ±1σ.

**Étape B :** Comparez avec une cross-validation 10-fold (`cross_val_score`, scoring = `'neg_root_mean_squared_error'`).  
Affichez :
- Le RMSE de chaque fold (barplot)
- La moyenne ± écart-type



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 4.1

Quelle est l'amplitude de variation du RMSE selon le `random_state` ?  
Cette variation est-elle négligeable ou significative par rapport à la valeur moyenne du RMSE ?  
Pourquoi la cross-validation est-elle une meilleure estimation que n'importe lequel des 30 splits ?

> ✏️ *Répondez ici.*

---

### 4.2 — Courbes d'apprentissage

Les courbes d'apprentissage permettent de diagnostiquer si votre modèle souffre de **biais** (underfitting) ou de **variance** (overfitting).

**Consigne :** Utilisez `learning_curve` de sklearn pour tracer les courbes d'apprentissage du modèle multiple :
- 15 tailles d'entraînement entre 5% et 100% du dataset
- 5-fold CV
- Scoring : RMSE
- Afficher les bandes d'incertitude (±1σ)

> `from sklearn.model_selection import learning_curve`  
> `train_sizes, train_scores, val_scores = learning_curve(pipe, X, y, cv=5, train_sizes=np.linspace(0.05, 1.0, 15), scoring='neg_root_mean_squared_error')`



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 4.2

Les courbes d'apprentissage convergent-elles ? Que cela indique-t-il sur le modèle ?  
Y a-t-il un signe d'overfitting ? D'underfitting ?  
Combien d'observations supplémentaires estimez-vous nécessaires pour significativement améliorer le modèle ?

> ✏️ *Répondez ici.*


---
## Partie 5 — Régularisation Ridge et Lasso

---

### 5.1 — Le problème : dataset enrichi avec du bruit

Le dataset original comporte 10 features pertinentes. Pour cette partie, vous allez utiliser `X_enrichi` — le même dataset enrichi de **15 features bruit** (colonnes `bruit_01` à `bruit_15`), pour un total de 25 features.

**Consigne :**

1. Splittez `X_enrichi` et `y` en train (80%) / test (20%), `random_state=42`
2. Entraînez un OLS classique (Pipeline `StandardScaler + LinearRegression`) sur ce dataset enrichi
3. Comparez ses performances (RMSE_train, RMSE_test, R²_test) avec le modèle de la Partie 2 (10 features seulement)



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 5.1

L'ajout de 15 features bruit améliore-t-il ou dégrade-t-il le RMSE_test ?  
Observez-vous un signe d'overfitting (gap entre train et test) ?  
Quelle métrique — R² brut ou R² ajusté — reflète mieux la situation réelle ?

> ✏️ *Répondez ici.*

---

### 5.2 — Ridge : chemin de régularisation

**Consigne :** Sur le dataset enrichi (25 features) :

1. Définissez une grille d'alphas : `alphas = np.logspace(-2, 5, 100)`
2. Pour chaque alpha, entraînez un Pipeline `StandardScaler + Ridge(alpha=alpha)` et calculez RMSE_train et RMSE_test
3. Produisez une figure avec **2 sous-graphiques** :
   - **Gauche** : chemin des coefficients Ridge en fonction de `log(alpha)` — colorez les 10 features réelles en bleu, les 15 features bruit en gris
   - **Droite** : courbe RMSE_train et RMSE_test en fonction de `log(alpha)` — marquez l'alpha optimal (min RMSE_test)

> 💡 `ridge.coef_` après `ridge.fit()` vous donne les coefficients pour chaque alpha



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 5.2

À mesure qu'alpha augmente, que se passe-t-il sur les coefficients des features bruit vs les features réelles ?  
Ridge annule-t-il des coefficients ? Qu'est-ce que cela implique pour la sélection de variables ?  
Quelle est la valeur d'alpha optimal ? Comment l'interpréter ?

> ✏️ *Répondez ici.*

---

### 5.3 — Lasso : sélection automatique de variables

**Consigne :** Sur le dataset enrichi (25 features) :

1. Utilisez `LassoCV(cv=10, max_iter=10000)` dans un Pipeline pour trouver automatiquement l'alpha optimal
2. Après l'entraînement, listez :
   - Le nombre de features avec coefficient non nul
   - Le nom des features conservées (seuil : `|coef| > 1e-6`)
   - Le nombre de features bruit incorrectement conservées
3. Comparez les RMSE_test de : OLS-25 / Ridge-optimal / Lasso-optimal


In [ ]:
# ✏️ VOTRE CODE ICI


**Graphique 3 — Barplot comparatif OLS vs Ridge vs Lasso**

Produisez un barplot horizontal avec **3 groupes** (un par modèle : OLS-25, Ridge, Lasso)  
montrant les coefficients de chaque feature.  
Pour Lasso, les features à coefficient nul doivent apparaître à 0 dans le graphique.  
Utilisez des couleurs différentes pour les 3 modèles.



In [ ]:
# ✏️ VOTRE CODE ICI


#### ❓ Question 5.3

Lasso a-t-il réussi à éliminer les features bruit ? Partiellement ? Totalement ?  
Parmi les features conservées par Lasso, y en a-t-il qui vous surprennent ?  
Quel modèle recommanderiez-vous si le médecin veut un modèle **interprétable** avec le moins de variables possible ?

> ✏️ *Répondez ici.*


---
##  Synthèse — Tableau comparatif et recommandation finale

---

### Tableau comparatif des modèles

Complétez ce tableau avec vos résultats numériques réels :

| Modèle | Features | RMSE_train | RMSE_test | R²_test | R²_adj_test | CV RMSE (±std) | Coefs ≠ 0 |
|--------|----------|-----------|----------|--------|------------|---------------|-----------|
| Régression simple (1 feature) | 1 | ??? | ??? | ??? | — | — | 1 |
| OLS multiple | 10 | ??? | ??? | ??? | ??? | ??? | 10 |
| OLS enrichi (25 features) | 25 | ??? | ??? | ??? | ??? | — | 25 |
| Ridge optimal | 25 | ??? | ??? | ??? | — | — | 25 |
| Lasso optimal | 25 | ??? | ??? | ??? | — | — | ??? |

> ✏️ *Remplacez tous les ??? par vos chiffres.*

---

### Recommandation finale

En tant que data scientist, vous devez présenter **une recommandation** au médecin responsable du projet.

**Répondez aux 4 questions suivantes en 2–3 phrases chacune :**

**1. Quel modèle recommandez-vous et pourquoi ?**  
*(Justifiez en termes de performance, d'interprétabilité et de robustesse.)*

> ✏️ *Votre recommandation.*

---

**2. Quelle est la précision réelle de votre modèle en conditions réelles ?**  
*(Utilisez le RMSE de la cross-validation — pas le RMSE sur le test unique — et traduisez-le en termes métier.)*

> ✏️ *Votre réponse.*

---

**3. Quelles sont les trois limites principales de votre modèle ?**  
*(Pensez aux hypothèses du modèle linéaire, aux problèmes détectés dans les résidus, à la taille du dataset…)*

> ✏️ *Votre réponse.*

---

**4. Quelle serait votre prochaine étape pour améliorer le modèle ?**  
*(Features polynomiales, transformations de variables, autre algorithme…)*

> ✏️ *Votre réponse.*

---




---
## 💡 Pistes d'approfondissement (si vous avez du temps)

**Piste A — Feature engineering**  
La feature `imc` influence-t-elle la progression différemment selon l'âge ?  
Créez un terme d'interaction `imc_x_age = imc * age` et ajoutez-le au modèle.  
Le RMSE_test s'améliore-t-il ?

**Piste B — Transformation de la cible**  
Appliquez `log(y)` comme cible (si y > 0). Entraînez le même modèle multiple.  
Comparez les résidus avant et après transformation logarithmique.

**Piste C — ElasticNet**  
Implémentez un `ElasticNetCV` avec `l1_ratio=[0.1, 0.5, 0.7, 0.9, 1.0]`.  
Ajoutez-le au tableau comparatif. Quelle valeur de `l1_ratio` est sélectionnée ?

**Piste D — Stabilité des coefficients Lasso**  
Entraînez Lasso 20 fois avec 20 `random_state` différents pour le split.  
Les mêmes features sont-elles toujours sélectionnées ?  
Que cela révèle-t-il sur la robustesse de la sélection de variables Lasso ?
